In [ ]:
from pathlib import Path
import os
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config/paper.json').is_file())
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import json
import pandas as pd
from finder_core.settings import load_settings
parameters = json.loads(Path('config/settings.json').read_text())
ticker = parameters['ticker']
settings = load_settings(root, ticker=ticker)

import numpy as np


In [ ]:
window_size = settings['window_size']
label_window_size = settings['label_window_size']
crash_percentile = settings['crash_percentile']
market_data = pd.read_csv(root / settings['paths']['ohlcv'], index_col='Date', parse_dates=True).sort_index()
destination = root / 'Data' / ticker / 'Market'
destination.mkdir(parents=True, exist_ok=True)


In [ ]:
def calculate_indicators(frame):
    result = frame.copy()
    result.columns = [str(column).lower() for column in result.columns]
    result = result.sort_index()
    required_columns = ['close', 'high', 'low', 'open', 'volume']
    missing_columns = [column for column in required_columns if column not in result.columns]
    if missing_columns:
        raise ValueError(f'Missing required market columns: {missing_columns}')
    for column in required_columns:
        result[column] = pd.to_numeric(result[column], errors='coerce')
    close = result['close']
    result['return'] = np.log(close).diff()
    high = result['high']
    low = result['low']
    open_price = result['open']
    volume = result['volume']
    epsilon = 1e-10
    regime_window = window_size

    def wilder(series, period):
        return series.ewm(alpha=1.0 / period, adjust=False).mean()
    sma20 = close.rolling(20).mean()
    sma50 = close.rolling(50).mean()
    sma200 = close.rolling(200).mean()
    standard_deviation_20d = close.rolling(20).std()
    result['sma20'] = sma20
    result['sma50'] = sma50
    result['sma200'] = sma200
    result['bb_upper'] = sma20 + 2 * standard_deviation_20d
    result['bb_lower'] = sma20 - 2 * standard_deviation_20d
    result['bb_width'] = (result['bb_upper'] - result['bb_lower']) / (sma20 + epsilon)
    result['death_cross'] = ((sma50.shift(1) > sma200.shift(1)) & (sma50 < sma200)).astype(int)
    result['golden_cross'] = ((sma50.shift(1) < sma200.shift(1)) & (sma50 > sma200)).astype(int)
    result['logreturn'] = np.log(close / close.shift(1))
    log_returns = result['logreturn']
    result['hv_126d_ann'] = log_returns.rolling(regime_window).std() * np.sqrt(252)
    result['hv_10d_ann'] = log_returns.rolling(10).std() * np.sqrt(252)
    result['hv_60d_ann'] = log_returns.rolling(60).std() * np.sqrt(252)
    log_high_low = np.log(high / low)
    log_close_open = np.log(close / open_price)
    result['hv_parkinson_126d_ann'] = np.sqrt(log_high_low.pow(2).rolling(regime_window).mean() / (4 * np.log(2))) * np.sqrt(252)
    gk_variance = 0.5 * log_high_low.pow(2).rolling(regime_window).mean() - (2 * np.log(2) - 1) * log_close_open.pow(2).rolling(regime_window).mean()
    result['gk_variance_126d'] = gk_variance
    result['hv_gk_126d_ann'] = np.sqrt(gk_variance.where(gk_variance > 0)) * np.sqrt(252)
    log_high_open = np.log(high / open_price)
    log_low_open = np.log(low / open_price)
    rs_variance = (log_high_open * (log_high_open - log_close_open) + log_low_open * (log_low_open - log_close_open)).rolling(regime_window).mean()
    result['rs_variance_126d'] = rs_variance
    result['hv_rs_126d_ann'] = np.sqrt(rs_variance.where(rs_variance > 0)) * np.sqrt(252)
    volatility_columns = ['hv_126d_ann', 'hv_parkinson_126d_ann', 'hv_gk_126d_ann', 'hv_rs_126d_ann']
    result['composite_vol'] = result[volatility_columns].mean(axis=1)
    result['vol_of_vol_21d'] = result['composite_vol'].rolling(21).std()
    for quantile in (0.01, 0.05, 0.95, 0.99):
        label = int(quantile * 100)
        result[f'log_return_1d_q{label:02d}_roll126d'] = log_returns.rolling(regime_window).quantile(quantile)
        result[f'hv_126d_ann_q{label:02d}_roll126d'] = result['hv_126d_ann'].rolling(regime_window).quantile(quantile)
    ema12 = close.ewm(span=12, adjust=False, min_periods=12).mean()
    ema26 = close.ewm(span=26, adjust=False, min_periods=26).mean()
    macd = ema12 - ema26
    macd_signal = macd.ewm(span=9, adjust=False, min_periods=9).mean()
    macd_histogram = macd - macd_signal
    result['macd_12_26_norm'] = macd / (close + epsilon)
    result['macd_signal_9_norm'] = macd_signal / (close + epsilon)
    result['macd_hist_12_26_9_norm'] = macd_histogram / (close + epsilon)
    result['roc_5d'] = (close / close.shift(5) - 1) * 100
    result['roc_10d'] = (close / close.shift(10) - 1) * 100
    result['roc_20d'] = (close / close.shift(20) - 1) * 100
    previous_close = close.shift(1)
    true_range = pd.concat([high - low, (high - previous_close).abs(), (low - previous_close).abs()], axis=1).max(axis=1)
    atr_14d = wilder(true_range, 14)
    result['tr_1d_points'] = true_range
    result['atr_14d_points'] = atr_14d
    result['tr_1d_norm'] = true_range / (close + epsilon)
    result['atr_14d_norm'] = atr_14d / (close + epsilon)
    result['tr_to_atr_14d'] = true_range / (atr_14d + epsilon)
    money_flow_multiplier = (close - low - (high - close)) / (high - low + epsilon)
    result['cmf_20d'] = (money_flow_multiplier * volume).rolling(20).sum() / volume.rolling(20).sum().replace(0, np.nan)
    volume_mean_20d = volume.rolling(20).mean()
    volume_standard_deviation_20d = volume.rolling(20).std()
    result['volume_z_20d'] = (volume - volume_mean_20d) / (volume_standard_deviation_20d + epsilon)
    signed_volume = np.sign(close.diff()) * volume
    result['obv_flow_20d_norm'] = signed_volume.rolling(20).sum() / volume.rolling(20).sum().replace(0, np.nan)
    result['obv'] = signed_volume.fillna(0).cumsum()
    lowest_low = low.rolling(14).min()
    highest_high = high.rolling(14).max()
    result['stoch_k'] = 100 * (close - lowest_low) / (highest_high - lowest_low + epsilon)
    result['stoch_d'] = result['stoch_k'].rolling(3).mean()
    result['williamsr'] = -100 * (highest_high - close) / (highest_high - lowest_low + epsilon)
    result['price_zscore_20d'] = (close - sma20) / (standard_deviation_20d + epsilon)
    delta = close.diff()
    average_gain = wilder(delta.clip(lower=0), 14)
    average_loss = wilder((-delta).clip(lower=0), 14)
    result['rsi_14d'] = 100 - 100 / (1 + average_gain / (average_loss + epsilon))
    typical_price = (high + low + close) / 3
    typical_price_mean = typical_price.rolling(20).mean()
    mean_deviation = typical_price.rolling(20).apply(lambda values: np.mean(np.abs(values - np.mean(values))), raw=True)
    result['cci_20d'] = (typical_price - typical_price_mean) / (0.015 * mean_deviation + epsilon)
    result['sharpe_20_ann_rf0'] = log_returns.rolling(20).mean() / (log_returns.rolling(20).std() + epsilon) * np.sqrt(252)
    result['dd_20'] = (close / close.rolling(20).max() - 1) * 100
    maximum_close_14d = close.rolling(14).max()
    drawdown_14d = (close - maximum_close_14d) / maximum_close_14d * 100
    result['ulcer_index'] = np.sqrt(drawdown_14d.pow(2).rolling(14).mean())

    def expected_shortfall(values, probability=0.05):
        finite_values = values[np.isfinite(values)]
        if len(finite_values) == 0:
            return np.nan
        tail_count = max(int(np.ceil(probability * len(finite_values))), 1)
        return np.sort(finite_values)[:tail_count].mean()
    result['es_5pct_126'] = log_returns.rolling(regime_window).apply(expected_shortfall, raw=True)
    upward_move = high.diff()
    downward_move = -low.diff()
    plus_dm = pd.Series(np.where((upward_move > downward_move) & (upward_move > 0), upward_move, 0.0), index=result.index)
    minus_dm = pd.Series(np.where((downward_move > upward_move) & (downward_move > 0), downward_move, 0.0), index=result.index)
    if ((plus_dm > 0) & (minus_dm > 0)).any():
        raise RuntimeError('Directional movement components overlap.')
    plus_di = 100 * wilder(plus_dm, 14) / (result['atr_14d_points'] + epsilon)
    minus_di = 100 * wilder(minus_dm, 14) / (result['atr_14d_points'] + epsilon)
    directional_index = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + epsilon)
    result['plus_di'] = plus_di
    result['minus_di'] = minus_di
    result['adx'] = wilder(directional_index, 14)
    result['skewness_126'] = log_returns.rolling(regime_window).skew()
    result['kurtosis_126'] = log_returns.rolling(regime_window).kurt()
    return result
indicator_columns_full = ['close', 'high', 'low', 'open', 'volume', 'return', 'sma20', 'sma50', 'sma200', 'bb_upper', 'bb_lower', 'bb_width', 'death_cross', 'golden_cross', 'logreturn', 'hv_126d_ann', 'hv_10d_ann', 'hv_60d_ann', 'hv_parkinson_126d_ann', 'gk_variance_126d', 'hv_gk_126d_ann', 'rs_variance_126d', 'hv_rs_126d_ann', 'composite_vol', 'vol_of_vol_21d', 'log_return_1d_q01_roll126d', 'hv_126d_ann_q01_roll126d', 'log_return_1d_q05_roll126d', 'hv_126d_ann_q05_roll126d', 'log_return_1d_q95_roll126d', 'hv_126d_ann_q95_roll126d', 'log_return_1d_q99_roll126d', 'hv_126d_ann_q99_roll126d', 'macd_12_26_norm', 'macd_signal_9_norm', 'macd_hist_12_26_9_norm', 'roc_5d', 'roc_10d', 'roc_20d', 'tr_1d_points', 'atr_14d_points', 'tr_1d_norm', 'atr_14d_norm', 'tr_to_atr_14d', 'cmf_20d', 'volume_z_20d', 'obv_flow_20d_norm', 'obv', 'stoch_k', 'stoch_d', 'williamsr', 'price_zscore_20d', 'rsi_14d', 'cci_20d', 'sharpe_20_ann_rf0', 'dd_20', 'ulcer_index', 'es_5pct_126', 'plus_di', 'minus_di', 'adx', 'skewness_126', 'kurtosis_126']


In [ ]:
calculated = calculate_indicators(market_data)
indicators = calculated[indicator_columns_full].replace([np.inf, -np.inf], np.nan).dropna()
log_returns = np.log(market_data['Close']).diff()
quantile_series = log_returns.rolling(label_window_size).quantile(crash_percentile).shift(1)
future_returns = log_returns.shift(-1)
labels = (future_returns <= quantile_series).astype(float).where(quantile_series.notna() & future_returns.notna())
label_data = pd.DataFrame({'crash_label': labels, 'min_future_logret': future_returns.where(quantile_series.notna()), 'crash_threshold': quantile_series})
indicators.to_csv(destination / 'df_Indicators.csv', index_label='Date')
label_data.to_csv(destination / 'df_Crash_Label.csv', index_label='date')
log_returns.to_frame('logreturn').to_csv(destination / 'df_Log_Returns.csv', index_label='Date')
quantile_series.to_frame('crash_quantile').to_csv(destination / 'df_Quantile_Series.csv', index_label='Date')
display(indicators.tail())
